### We will try to understand `Entity Framework core` query here.

API `company_name`

In [19]:
%%sql
SELECT id, company_name
FROM ( -- MIN did not worked in guid so text and then guid again
    SELECT MIN(id::text)::uuid as id, company_name
    FROM combined_price_history
    WHERE company_name ILIKE '%Hydro%'
    GROUP BY company_name
)
LIMIT 10;


,id,company_name
0,000a2f78-ee65-4caa-9a01-e32e189f910c,Ankhukhola Hydropower Company Limited
1,0006a4ca-3bf4-4607-b23c-6ec5f8644e83,Arun Valley Hydropower Development Company Lim...
2,007a281e-596c-400f-ac6e-8502271c4220,Asian Hydropower Limited
3,00e9242b-b7ee-40c1-854c-9c80ddbd2e82,Balephi Hydropower Limited
4,003acb6c-0730-4f14-8a46-6a2386bcc5a7,Barahi Hydropower Public Limited
5,0004a50e-7d3f-4e62-9bdf-91cad2a2953e,Barun Hydropower Company Limited
6,0098cdf2-7796-4e57-a7ae-0782dd0cb010,Bhagawati Hydropower Development Company Limited
7,011e0004-db96-4849-981d-823670e01895,Bikash Hydropower Company Limited
8,002638cd-d6f1-46a5-aa7c-90750437f9a2,Bindhyabasini Hydropower Development Company L...
9,00064561-ecd3-471b-a428-db1315fe722b,Buddha Bhumi Nepal Hydropower Company Limited


 API `company_price_history`

There is a conditional query in the API controller I am not going to write that here. It will be lengthy but the logic is similar if we understand one.

In [6]:
%%sql
SELECT
    id,
    high,
    low,
    ltp,
    company_name,
    qty,
    percent_change,
    date_added
FROM combined_price_history
    WHERE company_name = 'Global IME Laghubitta Bittiya Sanstha Limited'
    -- Instead of writing all the edge cases let's say we have both from and to date
    AND date_added >= '2023-11-07' AND date_added <= '2023-11-09' -- In the api the ORM executes SQL according to the condition



,id,high,low,ltp,company_name,qty,percent_change,date_added
0,ad42f63b-8a43-45f8-b007-9efb992884d7,870.0,774.0,870,Global IME Laghubitta Bittiya Sanstha Limited,2976,8.07,2023-11-08
1,e456300d-aa77-4f5f-abdc-2c2e35168f90,847.7,779.1,805,Global IME Laghubitta Bittiya Sanstha Limited,2959,0.75,2023-11-07
2,4e9a5d79-6347-4af0-8032-0d2cf58833bc,852.6,783.0,832,Global IME Laghubitta Bittiya Sanstha Limited,2541,-4.37,2023-11-09


### C# memory level operation in SQL.

In `Microsoft Entity Framework Core` the method `ToListAsync()` executes the SQL.
And in our API `company_price_history` the SQL like operation is happening in the memory.

If someone gets curious then they can send a get request to the endpoint `company_price_history`

In [9]:
%%sql

SELECT -- The following opreation happnes in language level in C# not in SQL.
    DISTINCT ON (date_added) -- This ensure unqiueness because the API to plot a chart in the client side will throw an error, it also expects the date_added to be in order.
    id,
    high,
    low,
    ltp,
    company_name,
    qty,
    percent_change
FROM combined_price_history
    WHERE company_name = 'Global IME Laghubitta Bittiya Sanstha Limited'
    AND date_added >= '2023-11-07' AND date_added <= '2023-11-09'
ORDER BY date_added;


,id,high,low,ltp,company_name,qty,percent_change
0,e456300d-aa77-4f5f-abdc-2c2e35168f90,847.7,779.1,805,Global IME Laghubitta Bittiya Sanstha Limited,2959,0.75
1,ad42f63b-8a43-45f8-b007-9efb992884d7,870.0,774.0,870,Global IME Laghubitta Bittiya Sanstha Limited,2976,8.07
2,4e9a5d79-6347-4af0-8032-0d2cf58833bc,852.6,783.0,832,Global IME Laghubitta Bittiya Sanstha Limited,2541,-4.37


### Sector with the highest market capitalization.

Let's see how this operation is handed in SQL as well as ef core.